# Step 2b — Corrected Synthetic Patient Generator

This is the corrected Version 1 generator for **bioprosthetic valve durability prediction**.

### Fixes compared with Step 2

1. **Event-time row is explicitly created** for patients who experience an event.
2. **Echo/hemodynamic missingness is realistic**, rather than assuming an echo at every timepoint.
3. **Observation probability depends partly on clinical severity**, so sicker patients are more likely to have selected follow-up measurements.

The reference `multimodal_patient_year.csv` is still used only to estimate empirical lab/medication distributions and missingness.

> Valve-specific hemodynamic values and progression rules remain synthetic assumptions for model prototyping and should be clinically reviewed before scientific interpretation.


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

REFERENCE_CSV = Path("multimodal_patient_year.csv")
OUTDIR = Path("synthetic_generator_outputs_v2")
OUTDIR.mkdir(parents=True, exist_ok=True)

N_PATIENTS = 1000
REGULAR_TIMEPOINT_MONTHS = np.arange(0, 121, 12)

print("Reference:", REFERENCE_CSV.resolve())
print("Output directory:", OUTDIR.resolve())


Reference: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\multimodal_patient_year.csv
Output directory: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs_v2


## 1. Load reference multimodal dataset

In [2]:
if not REFERENCE_CSV.exists():
    raise FileNotFoundError(
        f"Reference CSV not found: {REFERENCE_CSV.resolve()}\n"
        "Place multimodal_patient_year.csv next to this notebook or update REFERENCE_CSV."
    )

ref = pd.read_csv(REFERENCE_CSV)

LAB_COLS = [c for c in ref.columns if c.startswith("lab__")]
MED_COLS = [c for c in ref.columns if c.startswith("med__") and c.endswith("__present")]

print("Reference shape:", ref.shape)
print("Patients:", ref["Patient"].nunique())
print("Labs:", len(LAB_COLS))
print("Medication indicators:", len(MED_COLS))
display(ref.head())


Reference shape: (226, 82)
Patients: 117
Labs: 30
Medication indicators: 15


,Patient,Year,note_count,note_types,note_services,signed_statuses,provider_types,provider_specialties,min_creation_year,max_creation_year,...,med__antiarrhythmic__present,med__insulin__present,medications_observed,Index_Date,Valve_Failure,notes_available,labs_available,medications_available,rich_lab_med,relative_to_index_year_UNVERIFIED
0,Patient_001,2022,1.0,Operative Report,Cardiac Surgery,Signed,Physician,Cardiac Surg,2022.0,2022.0,...,NaN,NaN,0,2022.0,1.0,1.0,0.0,0.0,0.0,0.0
1,Patient_001,2025,1.0,Progress Notes,NaN,Signed,Physician,Cardiology,2025.0,2025.0,...,NaN,NaN,0,2022.0,1.0,1.0,0.0,0.0,0.0,3.0
2,Patient_002,2017,1.0,Operative Report,Cardiac Surgery,Signed,Physician,NaN,2017.0,2017.0,...,NaN,NaN,0,2018.0,1.0,1.0,0.0,0.0,0.0,-1.0
3,Patient_002,2018,1.0,Progress Notes,NaN,Signed,Physician Assistant,NaN,2018.0,2018.0,...,NaN,NaN,0,2018.0,1.0,1.0,0.0,0.0,0.0,0.0
4,Patient_003,2014,1.0,Operative Report,Cardiac Surgery,Signed,Physician,NaN,2014.0,2014.0,...,NaN,NaN,0,2014.0,0.0,1.0,0.0,0.0,0.0,0.0


## 2. Estimate reference distributions and observed rates

In [3]:
def numeric_reference_stats(df, columns):
    rows = []
    for c in columns:
        s = pd.to_numeric(df[c], errors="coerce")
        observed = s.dropna()
        if observed.empty:
            continue

        q25, med, q75 = observed.quantile([0.25, 0.50, 0.75])

        rows.append({
            "feature": c,
            "median": float(med),
            "q25": float(q25),
            "q75": float(q75),
            "iqr": float(q75 - q25),
            "min": float(observed.min()),
            "max": float(observed.max()),
            "observed_rate": float(s.notna().mean()),
        })

    return pd.DataFrame(rows)


lab_stats = numeric_reference_stats(ref, LAB_COLS)

med_stats = []
for c in MED_COLS:
    s = pd.to_numeric(ref[c], errors="coerce")
    observed = s.dropna()

    if observed.empty:
        continue

    med_stats.append({
        "feature": c,
        "prevalence": float(observed.mean()),
        "observed_rate": float(s.notna().mean()),
    })

med_stats = pd.DataFrame(med_stats)

display(lab_stats)
display(med_stats)


,feature,median,q25,q75,iqr,min,max,observed_rate
0,lab__ALT,19.0000,15.00000,31.87500,16.87500,8.0000,139.00,0.168142
1,lab__APTT,30.2000,28.20000,31.20000,3.00000,25.6000,71.40,0.092920
2,lab__AST,23.0000,18.37500,28.00000,9.62500,13.0000,121.00,0.168142
3,lab__Albumin,3.7250,3.10000,4.10000,1.00000,2.6000,4.60,0.168142
4,lab__Alkaline Phosphatase,62.5000,50.25000,79.50000,29.25000,33.0000,114.50,0.168142
5,lab__Anion Gap,11.0000,10.00000,12.00000,2.00000,7.0000,16.50,0.194690
6,lab__BUN,20.0000,17.00000,24.00000,7.00000,4.0000,93.00,0.194690
7,lab__CO2,26.0000,25.00000,27.00000,2.00000,22.0000,31.00,0.194690
8,lab__Calcium,8.9500,8.67500,9.46250,0.78750,8.1000,10.60,0.194690
9,lab__Chloride,102.0000,101.00000,104.00000,3.00000,93.0000,106.00,0.194690


,feature,prevalence,observed_rate
0,med__beta_blocker__present,0.4750,0.353982
1,med__ace_inhibitor__present,0.2375,0.353982
2,med__arb__present,0.2250,0.353982
3,med__arni__present,0.1125,0.353982
4,med__loop_diuretic__present,0.3500,0.353982
5,med__thiazide_diuretic__present,0.1125,0.353982
6,med__mra__present,0.0750,0.353982
7,med__anticoagulant__present,0.3625,0.353982
8,med__antiplatelet__present,0.3750,0.353982
9,med__statin__present,0.4375,0.353982


## 3. Synthetic cohort assumptions

Four latent deterioration phenotypes are used **only inside the generator**:

- stable
- slow deterioration
- rapid deterioration
- regurgitation dominant

The latent phenotype is removed from the model-ready dataset.


In [4]:
PHENOTYPE_PROBS = {
    "stable": 0.55,
    "slow_deterioration": 0.25,
    "rapid_deterioration": 0.10,
    "regurgitation_dominant": 0.10,
}

PROCEDURE_TYPES = ["TAVR", "SAVR"]
PROCEDURE_PROBS = [0.65, 0.35]

VALVE_MODELS = [
    "Synthetic_Model_A",
    "Synthetic_Model_B",
    "Synthetic_Model_C",
]
VALVE_MODEL_PROBS = [0.40, 0.35, 0.25]

VALVE_SIZES_MM = [19, 21, 23, 25, 26, 27, 29]
VALVE_SIZE_PROBS = np.array([0.05, 0.12, 0.20, 0.18, 0.18, 0.15, 0.12])
VALVE_SIZE_PROBS = VALVE_SIZE_PROBS / VALVE_SIZE_PROBS.sum()

EVENT_TIME_RULES = {
    "stable":                 {"event_prob": 0.08, "mean": 108, "sd": 18},
    "slow_deterioration":     {"event_prob": 0.55, "mean": 84,  "sd": 18},
    "rapid_deterioration":    {"event_prob": 0.85, "mean": 48,  "sd": 14},
    "regurgitation_dominant": {"event_prob": 0.60, "mean": 72,  "sd": 18},
}

BASELINE_HEMO = {
    "mean_gradient_mmHg": (10.0, 2.5),
    "peak_velocity_m_s": (2.0, 0.25),
    "effective_orifice_area_cm2": (1.8, 0.25),
    "regurgitation_grade": (0.4, 0.5),
}


## 4. Helper functions

In [5]:
def clipped_normal(mean, sd, low=None, high=None):
    x = float(rng.normal(mean, sd))

    if low is not None:
        x = max(x, low)
    if high is not None:
        x = min(x, high)

    return x


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def sample_phenotype():
    return rng.choice(
        list(PHENOTYPE_PROBS.keys()),
        p=list(PHENOTYPE_PROBS.values()),
    )


def generate_event_and_censor(phenotype):
    rule = EVENT_TIME_RULES[phenotype]

    has_event = rng.random() < rule["event_prob"]
    event_time = None

    if has_event:
        event_time = clipped_normal(
            rule["mean"],
            rule["sd"],
            low=18,
            high=120,
        )

    censor_time = float(rng.uniform(48, 120))

    if event_time is not None and event_time <= censor_time:
        return 1, event_time, censor_time

    return 0, None, censor_time


def patient_time_grid(event, event_time, censor_time):
    end_time = event_time if event == 1 else censor_time

    regular = [
        float(t)
        for t in REGULAR_TIMEPOINT_MONTHS
        if t <= end_time
    ]

    # FIX 1:
    # Create an exact event-time row if the event does not fall
    # exactly on one of the regular yearly timepoints.
    if event == 1 and event_time is not None:
        if not any(np.isclose(event_time, t) for t in regular):
            regular.append(float(event_time))

    return sorted(set(regular))


def hemodynamic_trajectory(phenotype, months, baseline):
    t_years = months / 12.0

    grad = baseline["mean_gradient_mmHg"] + 0.35 * t_years
    vel = baseline["peak_velocity_m_s"] + 0.025 * t_years
    eoa = baseline["effective_orifice_area_cm2"] - 0.015 * t_years
    reg = baseline["regurgitation_grade"] + 0.03 * t_years

    if phenotype == "slow_deterioration":
        grad += 0.9 * max(t_years - 2, 0) ** 1.35
        vel += 0.07 * max(t_years - 2, 0) ** 1.20
        eoa -= 0.055 * max(t_years - 2, 0) ** 1.25

    elif phenotype == "rapid_deterioration":
        grad += 2.0 * max(t_years - 1, 0) ** 1.35
        vel += 0.14 * max(t_years - 1, 0) ** 1.20
        eoa -= 0.10 * max(t_years - 1, 0) ** 1.20

    elif phenotype == "regurgitation_dominant":
        reg += 0.42 * max(t_years - 2, 0) ** 1.15
        grad += 0.25 * max(t_years - 3, 0)
        eoa -= 0.015 * max(t_years - 3, 0)

    grad += rng.normal(0, 1.2)
    vel += rng.normal(0, 0.08)
    eoa += rng.normal(0, 0.08)
    reg += rng.normal(0, 0.20)

    grad = float(np.clip(grad, 3, 60))
    vel = float(np.clip(vel, 0.8, 6.0))
    eoa = float(np.clip(eoa, 0.35, 3.0))
    reg_grade = int(np.clip(np.rint(reg), 0, 4))

    return grad, vel, eoa, reg_grade


def sample_lab_value(feature, months, hf_burden):
    row = lab_stats[lab_stats["feature"] == feature]

    if row.empty:
        return np.nan

    row = row.iloc[0]

    median = row["median"]
    scale = max(
        row["iqr"] / 1.349,
        abs(median) * 0.05,
        1e-6,
    )

    value = rng.normal(median, scale)
    name = feature.lower()
    t_years = months / 12.0

    if "nt-probnp" in name or "bnp" in name:
        value = max(
            0,
            value * (1 + 0.12 * hf_burden + 0.03 * t_years)
        )

    elif "creatinine" in name:
        value = max(
            0,
            value * (1 + 0.03 * hf_burden + 0.01 * t_years)
        )

    elif "egfr" in name:
        value = max(
            1,
            value * (1 - 0.025 * hf_burden - 0.01 * t_years)
        )

    elif "lvef" in name:
        value = value - 1.4 * hf_burden - 0.25 * t_years

    return float(np.clip(value, row["min"], row["max"]))


def medication_probability(feature, hf_burden, af, ckd):
    row = med_stats[med_stats["feature"] == feature]

    base = (
        float(row.iloc[0]["prevalence"])
        if not row.empty
        else 0.20
    )

    base = np.clip(base, 0.001, 0.999)
    logit = math.log(base / (1 - base))

    name = feature.lower()

    if "loop_diuretic" in name:
        logit += 0.9 * hf_burden

    if "anticoagulant" in name:
        logit += 1.5 * af

    if "sglt2" in name:
        logit += 0.35 * hf_burden + 0.20 * ckd

    if "beta_blocker" in name:
        logit += 0.25 * hf_burden + 0.25 * af

    return float(np.clip(sigmoid(logit), 0.01, 0.99))


def echo_observation_probability(months, hf_burden, is_event_row):
    # FIX 2 + 3:
    # Echo is not available at every timepoint.
    # Follow-up imaging becomes more likely with higher burden.
    base = 0.45

    if months == 0:
        base = 0.90

    if is_event_row:
        base = 0.98

    p = base + 0.12 * min(hf_burden, 2.5)

    return float(np.clip(p, 0.10, 0.99))


def lab_observation_probability(feature, reference_rate, hf_burden):
    # Start from empirical observation rate, then slightly increase
    # sampling in clinically sicker states.
    name = feature.lower()

    boost = 0.0

    if (
        "bnp" in name
        or "troponin" in name
        or "creatinine" in name
        or "egfr" in name
        or "lvef" in name
    ):
        boost = 0.08 * min(hf_burden, 2.5)

    return float(np.clip(reference_rate + boost, 0.01, 0.98))


## 5. Generate corrected synthetic longitudinal cohort

In [6]:
rows = []

for i in range(N_PATIENTS):
    patient_id = f"SYN_{i+1:05d}"
    phenotype = sample_phenotype()

    procedure_type = rng.choice(
        PROCEDURE_TYPES,
        p=PROCEDURE_PROBS,
    )

    valve_model = rng.choice(
        VALVE_MODELS,
        p=VALVE_MODEL_PROBS,
    )

    valve_size_mm = int(
        rng.choice(
            VALVE_SIZES_MM,
            p=VALVE_SIZE_PROBS,
        )
    )

    age_mean = 76 if procedure_type == "TAVR" else 68
    age_at_implant = int(
        np.clip(
            rng.normal(age_mean, 8),
            45,
            95,
        )
    )

    sex = rng.choice(
        ["Female", "Male"],
        p=[0.46, 0.54],
    )

    ckd = int(
        rng.random()
        < np.clip(
            0.18 + 0.006 * (age_at_implant - 60),
            0.05,
            0.55,
        )
    )

    af = int(
        rng.random()
        < np.clip(
            0.20 + 0.004 * (age_at_implant - 60),
            0.08,
            0.50,
        )
    )

    heart_failure = int(
        rng.random()
        < np.clip(
            0.18 + 0.10 * (phenotype != "stable"),
            0.10,
            0.55,
        )
    )

    diabetes = int(rng.random() < 0.28)

    event, event_time, censor_time = generate_event_and_censor(
        phenotype
    )

    followup_end = (
        event_time
        if event == 1
        else censor_time
    )

    time_grid = patient_time_grid(
        event,
        event_time,
        censor_time,
    )

    baseline = {
        "mean_gradient_mmHg": clipped_normal(
            *BASELINE_HEMO["mean_gradient_mmHg"],
            low=4,
            high=20,
        ),
        "peak_velocity_m_s": clipped_normal(
            *BASELINE_HEMO["peak_velocity_m_s"],
            low=1.2,
            high=3.0,
        ),
        "effective_orifice_area_cm2": clipped_normal(
            *BASELINE_HEMO["effective_orifice_area_cm2"],
            low=1.0,
            high=2.8,
        ),
        "regurgitation_grade": clipped_normal(
            *BASELINE_HEMO["regurgitation_grade"],
            low=0,
            high=2,
        ),
    }

    for months in time_grid:
        is_event_row = int(
            event == 1
            and event_time is not None
            and np.isclose(months, event_time)
        )

        grad, vel, eoa, reg_grade = hemodynamic_trajectory(
            phenotype,
            months,
            baseline,
        )

        stenotic_burden = np.clip(
            (grad - 12) / 20,
            0,
            2,
        )

        regurg_burden = reg_grade / 3.0

        hf_burden = float(
            np.clip(
                0.45 * heart_failure
                + 0.55 * stenotic_burden
                + 0.65 * regurg_burden,
                0,
                3,
            )
        )

        echo_observed = (
            rng.random()
            < echo_observation_probability(
                months,
                hf_burden,
                is_event_row,
            )
        )

        row = {
            "Patient": patient_id,
            "time_since_implant_months": float(months),

            "procedure_type": procedure_type,
            "valve_model": valve_model,
            "valve_size_mm": valve_size_mm,
            "valve_position": "aortic",

            "age_at_implant": age_at_implant,
            "sex": sex,

            "ckd": ckd,
            "af": af,
            "heart_failure": heart_failure,
            "diabetes": diabetes,

            "echo_observed": int(echo_observed),

            "generator_latent_phenotype": phenotype,

            "duration_months": float(followup_end),
            "event": int(event),
            "event_type": (
                "regurgitation_dominant"
                if event == 1
                and phenotype == "regurgitation_dominant"
                else "hemodynamic_deterioration"
                if event == 1
                else "censored"
            ),

            "is_event_row": int(is_event_row),
        }

        # Hemodynamics are only recorded when echo is observed.
        row["mean_gradient_mmHg"] = grad if echo_observed else np.nan
        row["peak_velocity_m_s"] = vel if echo_observed else np.nan
        row["effective_orifice_area_cm2"] = eoa if echo_observed else np.nan
        row["regurgitation_grade"] = reg_grade if echo_observed else np.nan

        # Generate labs and immediately apply observation mechanism.
        for lab in LAB_COLS:
            value = sample_lab_value(
                lab,
                months,
                hf_burden,
            )

            stat_row = lab_stats[
                lab_stats["feature"] == lab
            ]

            empirical_rate = (
                float(stat_row.iloc[0]["observed_rate"])
                if not stat_row.empty
                else 0.25
            )

            p_obs = lab_observation_probability(
                lab,
                empirical_rate,
                hf_burden,
            )

            row[lab] = (
                value
                if rng.random() < p_obs
                else np.nan
            )

        # Generate medications with empirical availability/missingness.
        for med in MED_COLS:
            p_present = medication_probability(
                med,
                hf_burden,
                af,
                ckd,
            )

            stat_row = med_stats[
                med_stats["feature"] == med
            ]

            observed_rate = (
                float(stat_row.iloc[0]["observed_rate"])
                if not stat_row.empty
                else 0.35
            )

            if rng.random() < observed_rate:
                row[med] = int(
                    rng.random() < p_present
                )
            else:
                row[med] = np.nan

        # Structured note-derived signals.
        row["note_dyspnea"] = int(
            rng.random()
            < sigmoid(
                -2.2 + 1.2 * hf_burden
            )
        )

        row["note_valve_dysfunction"] = int(
            rng.random()
            < sigmoid(
                -3.0
                + 0.11 * max(grad - 10, 0)
                + 0.70 * reg_grade
            )
        )

        # Explicit post-event/leakage QA fields.
        row["Valve_Failure"] = int(is_event_row)
        row["note_mentions_prosthetic_failure"] = int(
            is_event_row
            and rng.random() < 0.90
        )

        rows.append(row)


synthetic_full = pd.DataFrame(rows)

print("Rows:", len(synthetic_full))
print("Patients:", synthetic_full["Patient"].nunique())
print(
    "Event rows:",
    int(synthetic_full["is_event_row"].sum()),
)
display(synthetic_full.head())


Rows: 7169
Patients: 1000
Event rows: 201


,Patient,time_since_implant_months,procedure_type,valve_model,valve_size_mm,valve_position,age_at_implant,sex,ckd,af,...,med__statin__present,med__ccb__present,med__sglt2_inhibitor__present,med__digoxin__present,med__antiarrhythmic__present,med__insulin__present,note_dyspnea,note_valve_dysfunction,Valve_Failure,note_mentions_prosthetic_failure
0,SYN_00001,0.0,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,NaN,1.0,NaN,NaN,NaN,0.0,0,0,0,0
1,SYN_00001,12.0,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,NaN,0.0,NaN,NaN,0.0,0.0,0,0,0,0
2,SYN_00001,24.0,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,NaN,NaN,0.0,NaN,NaN,1.0,0,0,0,0
3,SYN_00001,36.0,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,0.0,NaN,NaN,NaN,0.0,NaN,0,0,0,0
4,SYN_00001,48.0,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,NaN,1.0,0.0,NaN,NaN,NaN,0,1,0,0


## 6. Build leakage-safe model-ready dataset

In [7]:
DROP_FROM_MODEL = [
    "generator_latent_phenotype",
    "Valve_Failure",
    "note_mentions_prosthetic_failure",
    "is_event_row",
]

model_ready = synthetic_full.drop(
    columns=[
        c
        for c in DROP_FROM_MODEL
        if c in synthetic_full.columns
    ]
).copy()

print("Full shape:", synthetic_full.shape)
print("Model-ready shape:", model_ready.shape)

assert "Valve_Failure" not in model_ready.columns
assert "generator_latent_phenotype" not in model_ready.columns
assert "note_mentions_prosthetic_failure" not in model_ready.columns

print("Leakage-safe checks passed.")


Full shape: (7169, 71)
Model-ready shape: (7169, 67)
Leakage-safe checks passed.


## 7. Save outputs

In [8]:
full_path = (
    OUTDIR
    / "synthetic_multimodal_patient_year_v2_FULL.csv"
)

model_path = (
    OUTDIR
    / "synthetic_multimodal_patient_year_v2_MODEL_READY.csv"
)

synthetic_full.to_csv(
    full_path,
    index=False,
)

model_ready.to_csv(
    model_path,
    index=False,
)

print("Saved FULL:")
print(full_path.resolve())

print("\nSaved MODEL READY:")
print(model_path.resolve())


Saved FULL:
C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs_v2\synthetic_multimodal_patient_year_v2_FULL.csv

Saved MODEL READY:
C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs_v2\synthetic_multimodal_patient_year_v2_MODEL_READY.csv


## 8. Encoder input specification

In [9]:
STATIC_FEATURES = [
    "procedure_type",
    "valve_model",
    "valve_size_mm",
    "valve_position",
    "age_at_implant",
    "sex",
    "ckd",
    "af",
    "heart_failure",
    "diabetes",
]

PHYSIOLOGY_FEATURES = [
    "mean_gradient_mmHg",
    "peak_velocity_m_s",
    "effective_orifice_area_cm2",
    "regurgitation_grade",
] + LAB_COLS

MEDICATION_FEATURES = MED_COLS

NOTE_DERIVED_FEATURES = [
    "note_dyspnea",
    "note_valve_dysfunction",
]

MASK_FEATURES = [
    "echo_observed",
]

TARGET_COLUMNS = [
    "duration_months",
    "event",
    "event_type",
]

encoder_spec = {
    "static_features": STATIC_FEATURES,
    "physiology_features": PHYSIOLOGY_FEATURES,
    "medication_features": MEDICATION_FEATURES,
    "note_derived_features_v1_optional": NOTE_DERIVED_FEATURES,
    "mask_features": MASK_FEATURES,
    "targets": TARGET_COLUMNS,
    "time_column": "time_since_implant_months",
    "patient_id_column": "Patient",
}

spec_path = (
    OUTDIR
    / "encoder_input_spec_v2.json"
)

with open(
    spec_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        encoder_spec,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(json.dumps(
    encoder_spec,
    indent=2,
    ensure_ascii=False,
))


{
  "static_features": [
    "procedure_type",
    "valve_model",
    "valve_size_mm",
    "valve_position",
    "age_at_implant",
    "sex",
    "ckd",
    "af",
    "heart_failure",
    "diabetes"
  ],
  "physiology_features": [
    "mean_gradient_mmHg",
    "peak_velocity_m_s",
    "effective_orifice_area_cm2",
    "regurgitation_grade",
    "lab__ALT",
    "lab__APTT",
    "lab__AST",
    "lab__Albumin",
    "lab__Alkaline Phosphatase",
    "lab__Anion Gap",
    "lab__BUN",
    "lab__CO2",
    "lab__Calcium",
    "lab__Chloride",
    "lab__Creatinine",
    "lab__Glucose",
    "lab__Hematocrit",
    "lab__Hemoglobin",
    "lab__INR",
    "lab__LVEF",
    "lab__MCH",
    "lab__MCHC",
    "lab__MCV",
    "lab__NT-proBNP",
    "lab__Platelets",
    "lab__Potassium",
    "lab__RBC",
    "lab__RDW",
    "lab__Sodium",
    "lab__Total Bilirubin",
    "lab__Total Protein",
    "lab__Troponin T",
    "lab__WBC",
    "lab__eGFR"
  ],
  "medication_features": [
    "med__beta_blocker__present

## 9. Sanity checks

In [10]:
patient_level = (
    synthetic_full
    .sort_values(
        ["Patient", "time_since_implant_months"]
    )
    .groupby("Patient")
    .first()
)

checks = {
    "n_patients": int(
        synthetic_full["Patient"].nunique()
    ),

    "n_rows": int(
        len(synthetic_full)
    ),

    "event_rate_patient_level": float(
        patient_level["event"].mean()
    ),

    "median_duration_months": float(
        patient_level["duration_months"].median()
    ),

    "median_rows_per_patient": float(
        synthetic_full
        .groupby("Patient")
        .size()
        .median()
    ),

    "event_rows": int(
        synthetic_full["is_event_row"].sum()
    ),

    "patients_with_event_row": int(
        synthetic_full.loc[
            synthetic_full["is_event_row"] == 1,
            "Patient",
        ].nunique()
    ),

    "echo_observed_rate": float(
        synthetic_full["echo_observed"].mean()
    ),
}

print(json.dumps(
    checks,
    indent=2,
))

assert (
    checks["event_rows"]
    == checks["patients_with_event_row"]
)

event_patients = int(
    patient_level["event"].sum()
)

assert (
    checks["patients_with_event_row"]
    == event_patients
)

print("\nEvent-time representation check passed.")


{
  "n_patients": 1000,
  "n_rows": 7169,
  "event_rate_patient_level": 0.201,
  "median_duration_months": 75.62379942503279,
  "median_rows_per_patient": 7.0,
  "event_rows": 201,
  "patients_with_event_row": 201,
  "echo_observed_rate": 0.5654903054819361
}

Event-time representation check passed.


In [11]:
# QA by latent phenotype — generator evaluation only.
qa = (
    synthetic_full
    .groupby("generator_latent_phenotype")
    .agg(
        patients=("Patient", "nunique"),
        rows=("Patient", "size"),
        event_rate=("event", "mean"),
        echo_observed_rate=("echo_observed", "mean"),
        mean_gradient=("mean_gradient_mmHg", "mean"),
        mean_regurgitation=("regurgitation_grade", "mean"),
        mean_eoa=("effective_orifice_area_cm2", "mean"),
    )
)

display(qa)


,patients,rows,event_rate,echo_observed_rate,mean_gradient,mean_regurgitation,mean_eoa
generator_latent_phenotype,,,,,,,
rapid_deterioration,113,640,0.698438,0.637500,16.421494,0.463235,1.531484
regurgitation_dominant,94,674,0.382789,0.614243,11.660829,1.449275,1.770893
slow_deterioration,254,1864,0.292918,0.561159,13.596154,0.525813,1.581082
stable,539,3991,0.023052,0.547732,10.949867,0.546661,1.754239


## 10. What to inspect before Step 3

Before building tensors/encoders, check:

- patient-level event rate,
- follow-up duration,
- number of timepoints per patient,
- whether each event patient has exactly one event row,
- echo observation rate,
- missingness of labs and medications,
- hemodynamic trajectories by phenotype,
- whether the model-ready dataset contains no explicit leakage fields.

If these look reasonable, **Step 3** can build:

`X_static`  
`X_physiology [N, T, F_phys]`  
`X_medications [N, T, F_med]`  
`masks [N, T, F]`  
`duration [N]`  
`event [N]`

followed by:

**Valve MLP + Physiology GRU + Medication GRU + Fusion + Survival Head**.
